# Mass budget — how it works

This is the explanation and reference for the `quicksat` mass budget: what the data structures are, why they are shaped the way they are, how units, harness and margins are handled, and where the tool stops.

It is deliberately not a tutorial. `sample/mass_budget.ipynb` works the same machinery through as a single example against a sample satellite — what to type, in what order, to get a budget out. This notebook sits behind it and says why each piece behaves as it does. In Diátaxis terms, `sample/` holds the tutorials and how-to guides; `docs/` holds the explanation and the reference.

It still runs, because an explanation that cannot be executed drifts from the code it describes. It runs against `docs/data/`, its own copy of the sample satellite's input files. The copy is deliberate: the figures quoted in the prose below are only true of the data that produced them, and reading `sample/data/` would let a tutorial retuned for its own reasons quietly falsify this notebook.

In [1]:
import os
import tempfile
from pathlib import Path

import pandas as pd

# Make the in-development quicksat package importable without installing it: walk up
# from the current directory to the repo root (the folder that holds the quicksat
# package) and switch to it. Works whether the notebook runs from docs/, the repo
# root, or the docs build.
here = Path.cwd()
repo_root = next(
    (p for p in (here, *here.parents) if (p / "quicksat" / "__init__.py").exists()),
    None,
)
if repo_root is None:
    raise RuntimeError("could not locate the quicksat repo root")
os.chdir(repo_root)

from quicksat import Q_, u
from quicksat.mass.budget import MassBudget

pd.set_option("display.float_format", lambda value: f"{value:,.3f}")



In [2]:
DATA = Path("docs") / "data"
budget = MassBudget.from_csv(DATA / "equipment.csv", DATA / "mass_budget_config.yaml")

## The data model

A mass budget is usually drawn as a tree: satellite, then platform and payload, then subsystems, then boxes. `quicksat` does not store one. The table is flat — one row per equipment item — and `location`, `responsibility` and `subsystem` are three ordinary columns on that row.

The reason is that the three are cross-cutting rather than nested. A star tracker can sit physically on the payload module while belonging to the platform team and reporting under ADCS; in the sample satellite one does. A tree has to privilege one of the three as its spine and then express the other two as side tables that have to be kept in step by hand. A flat table privileges none of them, and `groupby` gives all three views for the same cost — which is why every aggregation below reconciles to the same total without any reconciliation code.

The three axes are not equal in what they *do*, though:

| axis | role |
|---|---|
| `location` | the computation axis — carries the system margin, the harness fraction, and whether the hardware survives separation |
| `responsibility` | reporting only — who owns the item |
| `subsystem` | reporting only — which discipline the item belongs to |

Only `location` is looked up in the config, so only `location` can change a number. A typo in `subsystem` silently creates a new reporting group; a typo in `location` is an error at load time, because the config has nothing to say about it.

In [3]:
budget.eqpt_table.head(4)

,equipment_id,equipment_name,location,responsibility,subsystem,eqpt_mass,eqpt_margin,number_of_units,mass_class,comments,eqpt_total_mass
0,telescope,TMA telescope assembly,Payload,Payload,Instrument,62.000,20.000,1,equipment,,62.000
1,focal_plane,Focal plane assembly,Payload,Payload,Instrument,11.500,15.000,1,equipment,,11.500
2,payload_electronics,Video processing unit,Payload,Payload,OBDH,8.000,10.000,1,equipment,,8.000
3,payload_radiator,Payload radiator,Payload,Payload,Thermal,3.400,20.000,1,equipment,,3.400


### The working table

`eqpt_table` is the validated equipment list after loading — a copy, so nothing downstream can write back into the budget. Its columns:

| column | meaning |
|---|---|
| `equipment_id`, `equipment_name` | the id says *what* a thing is (`magnetorquer`), the name says *which one* (`ZARM MT30`) — neither alone identifies an item |
| `location`, `responsibility`, `subsystem` | the three axes |
| `eqpt_mass` | mass of one unit, in kg, whatever unit it was entered in |
| `eqpt_margin` | per-item contingency, percent |
| `number_of_units` | how many are flown |
| `mass_class` | `equipment` or `propellant` |
| `comments` | free text, carried through from the file |
| `eqpt_total_mass` | `eqpt_mass × number_of_units`, still before any margin |

Two columns are derived rather than typed: `eqpt_total_mass`, and `eqpt_mass` wherever the file used a unit other than kg. Two *rows* are derived as well — one harness row per location, appended at the bottom. Both are covered below.

Everything from here on is plain floats in kilograms. Units are a loading concern, not a computation concern, and are re-attached only at the outer edge where a query returns a figure.

## The equipment file

One CSV, one row per item, no nesting:

| column | meaning |
|---|---|
| `equipment_id` | short identifier, no whitespace (`star_tracker`) |
| `equipment_name` | full name, free text (`Jena Astro HP`) |
| `location` | where it sits — the computation axis |
| `responsibility` | who owns it — reporting only |
| `subsystem` | `ADCS`, `EPS`, ... — reporting only |
| `unit_mass` | mass of one unit, **with its unit** (`100 kg`, `750 g`) |
| `eqpt_margin` | per-item contingency, percent |
| `number_of_units` | how many are flown |
| `mass_class` | `equipment` or `propellant`; blank means `equipment` |
| `comments` | free text |

All ten columns must be present; the first eight must be non-empty. Each row is validated on its own by a Pydantic model, so a malformed file names the offending row and field rather than failing later inside the arithmetic — a wrong number is much harder to find once it has been summed.

The rules that are not obvious from the column list:

- **The three axes may not contain whitespace.** `Attitude Control` is rejected, `ADCS` is not. Grouping keys that can differ by a trailing space are a reliable source of two groups where one was meant.
- **`equipment_id` is unique within a location, not globally.** Physically distinct items often share a name — the sample satellite carries `mli` and `heaters` at both Platform and Payload — so the same id at two locations is fine, and the same id twice in one location is an error.
- **`number_of_units` may be zero.** It keeps a candidate item in the file, with its mass and its margin recorded, contributing nothing until it is flown.
- **`eqpt_margin` and `number_of_units` may not be negative.** Neither may a mass.
- **A blank `mass_class` means `equipment`.** Only propellant has to be declared.

In [4]:
HEADER = (
    "equipment_id,equipment_name,location,responsibility,subsystem,"
    "unit_mass,eqpt_margin,number_of_units,mass_class,comments"
)
WHEEL = "wheel,Rockwell RSI 45,Platform,Platform,ADCS,7.0 kg,10,4,equipment,"


# build a one-off budget from CSV rows, and report what the loader makes of it
def load_rows(label, *rows):
    path = Path(tempfile.mkdtemp()) / "equipment.csv"
    path.write_text("\n".join((HEADER, *rows)) + "\n")
    try:
        MassBudget.from_csv(path, DATA / "mass_budget_config.yaml")
        print(f"{label:24s} accepted")
    except ValueError as exc:
        text = " ".join(str(exc).replace(str(path), "equipment.csv").split())
        print(f"{label:24s} rejected - {text[:120]}")


load_rows("a plain row", WHEEL)
load_rows("zero units", WHEEL.replace(",10,4,", ",10,0,"))
load_rows("grams", WHEEL.replace("7.0 kg", "750 g"))
load_rows("blank mass class", WHEEL.replace(",equipment,", ",,"))
load_rows("watts in a mass column", WHEEL.replace("7.0 kg", "100 W"))
load_rows("negative mass", WHEEL.replace("7.0 kg", "-7.0 kg"))
load_rows("space in an axis", WHEEL.replace(",ADCS,", ",Attitude Control,"))
load_rows("unconfigured location", WHEEL.replace(",Platform,Platform,", ",Deck,Platform,"))
load_rows("same id, two locations", WHEEL, WHEEL.replace(",Platform,Platform,", ",Payload,Payload,"))
load_rows("same id, one location", WHEEL, WHEEL.replace("Rockwell RSI 45", "Rockwell RSI 68"))

a plain row              accepted
zero units               accepted
grams                    accepted
blank mass class         accepted
watts in a mass column   rejected - equipment.csv, row 2: 1 validation error for Equipment unit_mass Value error, Value must have mass dimensions, got '100 
negative mass            rejected - equipment.csv, row 2: 1 validation error for Equipment unit_mass Value error, Mass must not be negative, got '-7.0 kilog
space in an axis         rejected - equipment.csv, row 2: 1 validation error for Equipment subsystem String should match pattern '^\S+$' [type=string_patter
unconfigured location    rejected - Location 'Deck' is used in the equipment list but is missing from the budget config. Add it under 'locations:'.
same id, two locations   accepted
same id, one location    rejected - Duplicate (equipment_id, location) pairs: [{'equipment_id': 'wheel', 'location': 'Platform'}]


## Units

`unit_mass` is entered as text with its unit — `100 kg`, `750 g`, `2.4 lb` — and parsed with [pint](https://pint.readthedocs.io/) on load. That buys two things a bare float does not.

The first is a dimension check. `100 W` in a mass column is not a number that happens to be wrong; it is not a mass at all, and it is rejected at the row that contains it. The same goes for a bare `100`, which has no dimension: quicksat does not guess kilograms.

The second is that the file can be written in whatever unit the datasheet uses. Conversion happens once, at load, and `eqpt_mass` is canonical kilograms from then on. The sample satellite's IMU is entered as `750 g` for exactly this reason.

Inside the computation there are no quantities, only floats. Units could be carried through the table itself — [pint-pandas](https://github.com/hgrecco/pint-pandas) gives pandas a unit-aware column dtype, and it survives `groupby`, `concat` and the rest — but two things argue against it here, and speed is not really one of them (it costs about 1.3× on a table this size, which is a fraction of a millisecond).

The first is that a unit-aware column reports a bad cell as a bare `DimensionalityError` with no row attached. Validating each row on load instead is what lets a malformed file say *which* row and *which* field, which is the whole reason the check is worth having.

The second is that a unit lives on the column, not the cell. That suits the mass table, where every row is a mass, and fails the report of the data budget, whose one `value` column holds minutes, Mbit/s and GB on different rows. Keeping the arithmetic in plain floats means one convention across every budget rather than one per table shape.

So units are checked at the edge and reappear at the edge: every query returns a pint `Quantity` in kg, which can be converted for reporting.

One constraint follows from how pint works: every `Quantity` in quicksat must come from the shared registry, exposed as `quicksat.u` and `quicksat.Q_`. Pint refuses to combine quantities built by two different registries, so a `Quantity` made with a locally constructed `pint.UnitRegistry()` will not add to a budget figure.

In [5]:
imu = budget.eqpt_table.query("equipment_id == 'imu'")
print("entered as 750 g, stored as:", imu["eqpt_mass"].item(), "kg")

mass = budget.in_orbit_mass()
print("\nqueries return a Quantity:", repr(mass))
print("  in kg      ", f"{mass:~.2f}")
print("  in pounds  ", f"{mass.to('lb'):~.2f}")
print("  magnitude  ", mass.magnitude)

# the shared registry is what makes this work; a Quantity from another registry
# would refuse to combine with it
print("\nplus a fixed allowance:", f"{mass + Q_(2.5, 'kg'):~.2f}")
print("same registry:", mass._REGISTRY is u)

entered as 750 g, stored as: 0.75 kg

queries return a Quantity: <Quantity(470.62, 'kilogram')>
  in kg       470.62 kg
  in pounds   1037.54 lb
  magnitude   470.62

plus a fixed allowance: 473.12 kg
same registry: True


## The config file

The config is a YAML file with one entry per location. Everything the budget computes is keyed there, which is what keeps the derived harness rows unambiguous about which margin applies to them.

| setting | meaning | default |
|---|---|---|
| `system_margin` | percent, applied to the location's subtotal *after* the per-item margins | 0 |
| `harness_fraction` | percent of this location's equipment mass, before margin | 0 |
| `harness_margin` | contingency on the derived harness row, percent | 0 |
| `retained_in_orbit` | whether hardware here survives separation | `true` |

All four have defaults, so a location that needs nothing can be an empty entry — but it still has to *be* an entry. A location used in the CSV and missing from the config is an error, not a silent zero, because a silent zero understates the budget in the direction nobody checks.

`Launcher` is the one reserved name. It needs no entry and defaults to no margins, no harness, and `retained_in_orbit: false`. It is the only place where behaviour is attached to a name rather than to a setting, and it exists because every budget has exactly one such location and writing it out each time adds nothing.

In [6]:
print((DATA / "mass_budget_config.yaml").read_text())
print("configured locations:", budget.config.locations)
print("\nLauncher, unlisted but reserved:", budget.config.for_location("Launcher"))

# quicksat mass budget configuration
#
# Everything is keyed on location: it carries both the system margin and the
# harness parameters. `Launcher` is a reserved location name and needs no entry --
# it defaults to no margins, no harness, and being dropped at separation.

locations:
  Platform:
    system_margin: 20     # percent, applied to the location subtotal
    harness_fraction: 4   # percent of this location's equipment mass, before margin
    harness_margin: 25    # percent, the harness's own contingency
  Payload:
    system_margin: 20
    harness_fraction: 5
    harness_margin: 25

configured locations: {'Platform': LocationConfig(system_margin=20.0, harness_fraction=4.0, harness_margin=25.0, retained_in_orbit=True), 'Payload': LocationConfig(system_margin=20.0, harness_fraction=5.0, harness_margin=25.0, retained_in_orbit=True)}

Launcher, unlisted but reserved: system_margin=0.0 harness_fraction=0.0 harness_margin=0.0 retained_in_orbit=False


## Margins

There are two margin layers, and they apply at different levels for different reasons.

**Equipment margin** is per item, from the `eqpt_margin` column. It is the contingency on that item's own mass estimate, and it belongs to the item because its size is a property of how well that item is known — a flight-proven wheel with a datasheet mass carries less than a payload that exists as a sketch. quicksat does not compute it from a maturity code; you enter the number you want.

**System margin** is per location, from the config. It covers what is not in the equipment list at all: the items nobody has thought of yet. It is applied to the location subtotal *after* the per-item margins, so it compounds with them rather than replacing them. It lives on location because it is a design-authority quantity — the platform and payload each carry their own — and because there is no defensible way to split "the things we forgot" across subsystems. That is why `subsystem_mass` carries no system margin at all.

**Propellant is never margined**, whatever the flags say. Propellant uncertainty is defined as a delta-V margin rather than as mass contingency, so margining the mass as well would double-count it. The propellant row in the sample file carries `eqpt_margin` of 0 and would be left alone even if it did not.

The two flags are independent multipliers on each row rather than a choice between three precomputed columns, so every combination is expressible — including the unusual one of a system margin without the equipment margins beneath it.

In [7]:
layers = pd.DataFrame(
    {
        "raw": budget.by_subsystem(eqpt_margin=False, sys_margin=False)["mass"],
        "+ eqpt margin": budget.by_subsystem(sys_margin=False)["mass"],
        "+ sys margin": budget.by_subsystem()["mass"],
    }
)
layers["growth %"] = (layers["+ sys margin"] / layers["raw"] - 1) * 100
layers

,raw,+ eqpt margin,+ sys margin,growth %
subsystem,,,,
ADCS,36.050,39.498,47.397,31.476
COMM,11.100,12.305,14.766,33.027
EPS,45.300,52.335,62.802,38.636
Harness,13.314,16.642,19.971,50.000
Instrument,73.500,87.625,105.150,43.061
OBDH,18.900,20.695,24.834,31.397
Propulsion,32.200,33.660,35.992,11.776
Structure,95.000,113.100,135.720,42.863
Thermal,16.800,19.990,23.988,42.786


In [8]:
# propellant is untouched by either margin flag, and by the system margin applied
# to the Platform location it sits in
print("propellant, at face value        ", f"{budget.propellant_mass():~.2f}")
for label, flags in [
    ("both margins", {}),
    ("no equipment margin", {"eqpt_margin": False}),
    ("no margins at all", {"eqpt_margin": False, "sys_margin": False}),
]:
    wet = budget.in_orbit_mass(**flags)
    dry = budget.in_orbit_mass(propellant=0, **flags)
    print(f"{label:33s} wet - dry = {(wet - dry):~.2f}")

propellant, at face value         22.00 kg
both margins                      wet - dry = 22.00 kg
no equipment margin               wet - dry = 22.00 kg
no margins at all                 wet - dry = 22.00 kg


## Harness

Harness is not entered by hand. Cabling mass is not known item by item at this stage of a design. Instead one row per location is derived, as `harness_fraction` of that location's equipment mass, and injected into the table before any aggregation happens — so it is a normal row from then on and every view picks it up without special-casing.

Three choices are worth being explicit about, because each one moves the number:

- **The base excludes propellant.** Cabling scales with the boxes it connects, not with the tank's contents.
- **The base is the mass before margin.** Using the margined mass would compound the equipment contingencies into the harness estimate, which is contingency on contingency.
- **The harness carries its own contingency**, `harness_margin`, usually larger than a box's — a derived fraction is a weaker estimate than a datasheet figure. In the sample config it is 25% against 5–20% on equipment.

The derived row takes its **location's name as its `responsibility`**, so that summing on responsibility includes it rather than dropping it on the floor. Where a location hosts several responsibilities that attributes the whole harness to the location's own name, which is a real limitation — the harness cannot be split across teams. Its `subsystem` stays `Harness`, so it remains a visible line of its own in the subsystem view rather than disappearing into a structure total.

A location with `harness_fraction: 0` gets no row at all, rather than a zero-mass one.

In [9]:
frame = budget.eqpt_table
harness = frame[frame["subsystem"] == "Harness"]
harness[["equipment_id", "location", "responsibility", "eqpt_total_mass", "eqpt_margin", "comments"]]

,equipment_id,location,responsibility,eqpt_total_mass,eqpt_margin,comments
31,harness_payload,Payload,Payload,5.200,25.000,Derived: 5.0% of 104.000 kg equipment mass
32,harness_platform,Platform,Platform,8.114,25.000,Derived: 4.0% of 202.850 kg equipment mass


In [10]:
# the derivation, reproduced by hand from the table and the config
for location, settings in budget.config.locations.items():
    base = frame[
        (frame["location"] == location)
        & (frame["mass_class"] == "equipment")
        & (frame["subsystem"] != "Harness")
    ]["eqpt_total_mass"].sum()
    derived = base * settings.harness_fraction / 100.0
    stored = harness.loc[harness["location"] == location, "eqpt_total_mass"].item()
    print(
        f"{location:9s} {settings.harness_fraction:4.1f}% of {base:8.3f} kg equipment"
        f"  = {derived:6.3f} kg   (stored: {stored:6.3f} kg)"
        f"  -> {derived * (1 + settings.harness_margin / 100):6.3f} kg with its"
        f" {settings.harness_margin:.0f}% margin"
    )

Platform   4.0% of  202.850 kg equipment  =  8.114 kg   (stored:  8.114 kg)  -> 10.142 kg with its 25% margin
Payload    5.0% of  104.000 kg equipment  =  5.200 kg   (stored:  5.200 kg)  ->  6.500 kg with its 25% margin


## Separation, and the four mass cases

Whether hardware survives separation is a property of its location, through `retained_in_orbit`. The `Launcher` location is the one that does not survive, and the `in_orbit` flag decides whether those rows are counted.

The separation interface itself is entered as two ordinary equipment rows — the satellite-side half at `location: Platform`, the launcher-side half at `location: Launcher` — each with its real mass. There is no split factor to configure, and an asymmetric interface, which is the normal case, costs nothing extra to express. The difference between the on-ground and in-orbit masses is then exactly the launcher-side rows, with their own margins applied.

Two of the query flags give the four masses normally quoted for a satellite:

| | `propellant=100` | `propellant=0` |
|---|---|---|
| `in_orbit=False` | launch mass | dry mass at launch |
| `in_orbit=True` | separated wet mass | in-orbit dry mass |

`propellant` is a percentage rather than a switch, because those two ends are not the only points of interest. Propellant rows are *scaled* rather than filtered out, so they stay visible in every grouping at whatever fraction is left, and a mass can be asked for at any point in the mission. The value is range-checked in `resolve()`, through which every query funnels.

In [11]:
print(f"launch mass (on ground, wet)   {budget.on_ground_mass():~.2f}")
print(f"dry mass at launch             {budget.on_ground_mass(propellant=0):~.2f}")
print(f"separated wet mass (in orbit)  {budget.in_orbit_mass():~.2f}")
print(f"in-orbit dry mass              {budget.in_orbit_mass(propellant=0):~.2f}")

left_behind = budget.on_ground_mass() - budget.in_orbit_mass()
launcher_rows = frame[frame["location"] == "Launcher"]
by_hand = (launcher_rows["eqpt_total_mass"] * (1 + launcher_rows["eqpt_margin"] / 100)).sum()
print(f"\ndropped at separation          {left_behind:~.2f}  (by hand: {by_hand:.3f} kg)")
print("  = the launcher-side adapter ring half and the clampband, with their margins")
print("  note the Launcher location carries no system margin, being unconfigured")

launch mass (on ground, wet)   487.34 kg
dry mass at launch             465.34 kg
separated wet mass (in orbit)  470.62 kg
in-orbit dry mass              448.62 kg

dropped at separation          16.72 kg  (by hand: 16.720 kg)
  = the launcher-side adapter ring half and the clampband, with their margins
  note the Launcher location carries no system margin, being unconfigured


## The query surface

Every query takes the same four flags, all defaulting to the same case, the satellite as it flies at the start of life — full propellant, both margins, after separation — so the usual question is a bare call and each deviation is one explicit switch.

| flag | meaning |
|---|---|
| `propellant` | percentage of the propellant load counted: 100 at start of life, 0 at end |
| `sys_margin` | apply the location's system margin |
| `eqpt_margin` | apply the per-item equipment margin |
| `in_orbit` | drop hardware at locations that do not survive separation |

`total_mass` is the generic query; everything else is a preset over it with a filter applied, so there is one place where a flag combination turns into a number. Three of the presets take fewer flags, and the missing ones are absent rather than ignored:

- `platform_mass` and `payload_mass` take no `in_orbit`, because retention is a property of the location and both are retained — the flag could not change the answer. They take a `by_responsibility` switch instead, because Platform and Payload each name both a location *and* a responsibility, and the two axes disagree wherever an item sits on one and belongs to the other.
- `subsystem_mass` takes no `sys_margin`, for the reason given above, and no `propellant`: a propellant row carries its own `subsystem`, so a Propulsion query picks the load up on its own and every other subsystem is unaffected.
- `propellant_mass` takes no flags at all. Propellant is never margined and is present both on the ground and in orbit, so there is nothing left for a flag to change.

In [12]:
print(f"everything, before system margin  {budget.total_mass(sys_margin=False):~.2f}")
print(f"half-way through the mission      {budget.in_orbit_mass(propellant=50):~.2f}")
print()
print(f"platform, by location             {budget.platform_mass():~.2f}")
print(f"platform, by responsibility       {budget.platform_mass(by_responsibility=True):~.2f}")
print(f"payload,  by location             {budget.payload_mass():~.2f}")
print(f"payload,  by responsibility       {budget.payload_mass(by_responsibility=True):~.2f}")
print("  the two axes differ by the star tracker: on the payload, the platform team's")
print()
print(f"one subsystem                     {budget.subsystem_mass('ADCS'):~.2f}")
print(f"propellant                        {budget.propellant_mass():~.2f}")

everything, before system margin  395.85 kg
half-way through the mission      459.62 kg

platform, by location             315.21 kg
platform, by responsibility       318.24 kg
payload,  by location             155.41 kg
payload,  by responsibility       152.38 kg
  the two axes differ by the star tracker: on the payload, the platform team's

one subsystem                     39.50 kg
propellant                        22.00 kg


## Aggregation

The same rows can be summed over any of the three axes. Each view takes the same flags, and returns a DataFrame indexed by the axis value with a single `mass` column.

All three reconcile to the same total for any flag combination, and not because anything reconciles them: they are three `groupby` calls over one resolved table. `location` happens also to be the computation axis, so the per-location view is the one that lines up with the margins and the harness, while the other two are pure cuts.

In [13]:
for flags in ({}, {"propellant": 0}, {"sys_margin": False}, {"in_orbit": False}):
    totals = {
        axis: view(**flags)["mass"].sum()
        for axis, view in [
            ("location", budget.by_location),
            ("responsibility", budget.by_responsibility),
            ("subsystem", budget.by_subsystem),
        ]
    }
    spread = max(totals.values()) - min(totals.values())
    print(f"{str(flags) or '{}  (defaults)':28s} {totals['location']:9.3f} kg   spread {spread:.1e}")

{}                             470.620 kg   spread 5.7e-14
{'propellant': 0}              448.620 kg   spread 5.7e-14
{'sys_margin': False}          395.850 kg   spread 5.7e-14
{'in_orbit': False}            487.340 kg   spread 0.0e+00


In [14]:
pd.concat(
    {"by location": budget.by_location(), "by responsibility": budget.by_responsibility()},
    axis=1,
)

,by location,by responsibility
,mass,mass
Payload,155.406,152.382
Platform,315.214,318.238


## The budget as a document

`tabulated_mass()` answers a different kind of question from the rest: not "how heavy is it" but "show me the budget". Every item, grouped into subsystem blocks within each location, then the location subtotal before its system margin, the margin itself, and the location total after it — followed by the dry mass, the propellant, and the wet mass.

Each row reads as the same three-step progression whatever level it sits at: `total_mass`, then `margin_pct`, then `total_mass_with_margin`. On an item row that margin is the equipment contingency; on a location total it is the system margin applied to the subtotal above. That is why there is no separate system-margin column or row — it is the same progression one level up.

Propellant appears once, at the bottom, and is held out of the blocks above it, so every subtotal on the way down is a dry mass and the column adds up as it reads. One consequence worth knowing: the Propulsion subsystem subtotal in this table is dry, while `subsystem_mass("Propulsion")` is wet. They answer different questions.

What comes back is a pandas `Styler`, not a frame: cells that do not apply to a row come out blank rather than `NaN`, masses print at a fixed number of decimals, and summary lines are bold. The frame is still there as `.data`, and it is complete — three columns are hidden from the rendered table but present in the data:

| column | why it is hidden |
|---|---|
| `location` | every subtotal and total row already names the location it closes |
| `comments` | free text stretches the table; worth turning on to watch the harness rows explain themselves |
| `row_type` | machine-facing: `equipment`, `subsystem_subtotal`, `location_subtotal`, `location_total`, `dry_total`, `propellant`, `wet_total` |

`row_type` is what makes the report filterable or exportable — the summary rows can be pulled out of `.data` without parsing captions. Per-subsystem subtotal lines are off by default because on a table this size they crowd out the items; the grouping by subsystem stays either way.

Locations and subsystems come out in order of first appearance in the equipment file rather than sorted, so the report keeps the structure the file was written with — and the derived harness, appended last, lands at the foot of its location.

In [15]:
report = budget.tabulated_mass(in_orbit=True)
report

Item,Name,Subsystem,Units,Mass [kg],Total Mass [kg],Margin [%],Total + Margin [kg]
telescope,TMA telescope assembly,Instrument,1,62.00,62.00,20,74.40
focal_plane,Focal plane assembly,Instrument,1,11.50,11.50,15,13.22
payload_electronics,Video processing unit,OBDH,1,8.00,8.00,10,8.80
payload_radiator,Payload radiator,Thermal,1,3.40,3.40,20,4.08
mli,Multi-layer insulation,Thermal,1,3.50,3.50,20,4.20
heaters,Heater lines and thermistors,Thermal,1,1.20,1.20,15,1.38
optical_bench,Optical bench,Structure,1,12.00,12.00,20,14.40
star_tracker,Jena Astro HP,ADCS,2,1.20,2.40,5,2.52
harness_payload,Payload harness,Harness,1,5.20,5.20,25,6.50
,Payload subtotal (before system margin),,,,109.20,,129.50


In [16]:
data = budget.tabulated_mass(subsystem_subtotals=True).data
print(data["row_type"].value_counts().to_dict())

summary = data[data["row_type"].isin(["dry_total", "propellant", "wet_total"])]
print()
columns = ["name", "total_mass", "total_mass_with_margin"]
print(summary[columns].to_string(index=False))  # pyright: ignore[reportAttributeAccessIssue]

propulsion = data.query("row_type == 'subsystem_subtotal' and subsystem == 'Propulsion'")
print()
print(f"Propulsion subtotal in the table  {propulsion['total_mass_with_margin'].item():.3f} kg  (dry)")
print(f"subsystem_mass('Propulsion')      {budget.subsystem_mass('Propulsion'):~.3f}  (wet)")

{'equipment': 30, 'subsystem_subtotal': 14, 'location_subtotal': 2, 'location_total': 2, 'dry_total': 1, 'propellant': 1, 'wet_total': 1}

                               name  total_mass  total_mass_with_margin
Total Dry Mass (with system margin)     373.850                 448.620
                         Propellant      22.000                  22.000
                     Total Wet Mass     395.850                 470.620

Propulsion subtotal in the table  11.660 kg  (dry)
subsystem_mass('Propulsion')      33.660 kg  (wet)


## Limitations

What the mass budget deliberately does not do, in rough order of how often it comes up:

- **Mass only.** No centre of gravity, no inertia tensor, no envelope or dimensions. `location` is a grouping key, not a coordinate.
- **No mass history.** The budget is whatever the files say today. Tracking growth across design reviews means versioning the CSV yourself.
- **Margins are entered, not derived.** There is no maturity or heritage code that turns into a contingency percentage, and no statistical combination — margins add linearly rather than in RSS, which is conservative and is meant to be.
- **One system margin per location, applied flat.** It cannot be varied by subsystem, and by construction cannot be attributed to one, so `subsystem_mass` and the subsystem view carry equipment margins only.
- **Harness is one derived row per location.** It cannot be split across responsibilities or subsystems, and it scales with equipment mass rather than with anything about routing or harness length.
- **Propellant is a single linear percentage of the load.** Residuals, unusable propellant, pressurant and blow-down behaviour are not modelled; each has to be entered as its own row if it matters. Nor is the load sized from a delta-V requirement — that is the delta-V module's job, and it is not written yet.
- **The reporting axes are free text.** `subsystem` and `responsibility` are validated for shape, not against a list, so a misspelling quietly produces an extra group rather than an error. Only `location` is checked, because only `location` is looked up in the config.
- **Nothing is written back.** The budget is built from the files and queried in memory; there is no export path that round-trips to CSV.